In [1]:
import numpy as np
import csv
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import preprocessing
from tensorflow.keras.layers import Normalization, Resizing
from tensorflow.keras import layers
from tensorflow.keras import models
from sklearn.utils.class_weight import compute_class_weight

import pandas as pd

import seaborn as sns

from scipy import signal
from scipy.signal import butter, filtfilt, iirnotch, periodogram

In [2]:
import os
cwd = os.getcwd()
data_path = str(os.path.join(cwd, 'data'))
print(data_path)

c:\Users\Aryan M\Documents\GitHub\ecz-ware\training\data


In [ ]:
SAMPLING_RATE = 1000
WIN_SIZE = 300
TIMESTEPS = 30
OVERLAP = 150

In [11]:
def normalize(data):
    return (data - np.mean(data)) / (np.std(data))

def bandpass_filter(data, lowcut=5.0, highcut=200.0, fs=1000.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, data)

def find_noise_frequency(data, fs=1000.0):
    f, Pxx = periodogram(data, fs=fs)
    idx = np.argmax(Pxx)
    return f[idx]

def adaptive_notch_filter(data, fs=1000.0, quality=30):
    freq = find_noise_frequency(data, fs=fs)
    nyq = 0.5 * fs
    b, a = iirnotch(freq/nyq, quality)
    return filtfilt(b, a, data)

def cleanup(data):
    data = normalize(data)
    data = bandpass_filter(data)
    data = adaptive_notch_filter(data)
    return data

In [17]:
def get_sample(data, start, win_size=200):
    end = start + win_size
    return data[start:end]

def get_data(filepath):
    df = pd.read_csv(filepath)
    return df[['v1', 'v2', 'label']].to_numpy()

def clean_data(data: np.ndarray):
    v1 = data[:, 0]
    v2 = data[:, 1]
    labels = data[:, 2]

    v1_cleaned = cleanup(v1)
    v2_cleaned = cleanup(v2)

    return np.column_stack((v1_cleaned, v2_cleaned, labels))

def split_data(data: np.ndarray, split_ratio=0.8):
    num_samples = data.shape[0]
    split_index = int(num_samples * split_ratio)
    
    train_data = data[:split_index]
    test_data = data[split_index:]
    
    return train_data, test_data

def zc(x, threshold=0.01):
    """Zero Crossing Rate with threshold to reduce noise influence"""
    return np.sum(((x[:-1] * x[1:]) < 0) & (np.abs(x[:-1] - x[1:]) > threshold))

def ssc(x, threshold=0.01):
    """Slope Sign Changes with threshold"""
    diff1 = np.diff(x)
    return np.sum(((diff1[:-1] * diff1[1:]) < 0) &
                    (np.abs(diff1[:-1] - diff1[1:]) > threshold))


def get_features(data: np.ndarray, verbose=False): # please feed in cleaned data
    ch1 = data[:, 0]
    ch2 = data[:, 1]
    
    labels = data[:, 2]

    all_labels = []
    all_features = []

    win_len = WIN_SIZE // TIMESTEPS

    for i in range(0, len(ch1) - WIN_SIZE + 1, OVERLAP):
        seq_feat = []
        if verbose:
            print("collecting features from {} to {}".format(i, i + OVERLAP))
        for j in range(TIMESTEPS):
            s = i + j * win_len
            
            sample_1 = get_sample(ch1, s, win_len)
            sample_2 = get_sample(ch2, s, win_len)

            mean_1 = np.mean(sample_1)
            mean_2 = np.mean(sample_2)

            std_1 = np.std(sample_1)
            std_2 = np.std(sample_2)

            wl_1 = np.sum(np.abs(np.diff(sample_1)))
            wl_2 = np.sum(np.abs(np.diff(sample_2)))

            zc_1 = zc(sample_1)
            zc_2 = zc(sample_2)

            ssc_1 = ssc(sample_1)
            ssc_2 = ssc(sample_2)

            feat1 = [mean_1, std_1, wl_1, zc_1, ssc_1]
            feat2 = [mean_2, std_2, wl_2, zc_2, ssc_2]

            feats = feat1 + feat2
            seq_feat.append(feats)
            
        all_features.append(seq_feat)
        all_labels.append(tf.cast(labels[i], tf.int32))


    return tf.stack(all_features, axis=0), tf.stack(all_labels, axis=0)


In [ ]:
scratch_data = np.vstack([
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_scratching_raina.csv')),
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_scratching_ariel.csv')),])

rest_data = np.vstack([
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_rest_raina.csv')),
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_rest_ariel.csv')),])

other_data = np.vstack([
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_other_raina.csv')),
    get_data(os.path.join(data_path+'\\training_set_15-07-2025\\emg_other_ariel.csv')),])

scratch_data = clean_data(scratch_data)
rest_data = clean_data(rest_data)
other_data = clean_data(other_data)

print(scratch_data)

[[ 1.19786804 -0.11554254  1.        ]
 [ 1.21013483 -0.22147576  1.        ]
 [ 1.46802936 -0.29568543  1.        ]
 ...
 [ 0.09662729 -1.8934858   1.        ]
 [ 0.06212717 -1.50062943  1.        ]
 [ 0.00309919 -0.29978867  1.        ]]
(5335, 80, 10) (5335,)


In [ ]:
df_scratch_train, df_scratch_test = split_data(scratch_data)
df_rest_train, df_rest_test = split_data(rest_data)
df_other_train, df_other_test = split_data(other_data)

feat_scratch_train, scratch_labels = get_features(df_scratch_train)
feat_rest_train, rest_labels = get_features(df_rest_train)
feat_other_train, other_labels = get_features(df_other_train)

feat_scratch_test, scratch_labels_test = get_features(df_scratch_test)
feat_rest_test, rest_labels_test = get_features(df_rest_test)
feat_other_test, other_labels_test = get_features(df_other_test)

print(feat_scratch_train.shape, scratch_labels.shape)

In [19]:
feat_all_train = tf.concat([feat_scratch_train, feat_rest_train, feat_other_train], axis=0)
label_all_train = tf.concat([scratch_labels, rest_labels, other_labels], axis=0)

feat_all_test = tf.concat([feat_scratch_test, feat_rest_test, feat_other_test], axis=0)
label_all_test = tf.concat([scratch_labels_test, rest_labels_test, other_labels_test], axis=0)

ds_scratch_train = tf.data.Dataset.from_tensor_slices((feat_scratch_train, scratch_labels))
ds_rest_train = tf.data.Dataset.from_tensor_slices((feat_rest_train, rest_labels))
ds_other_train = tf.data.Dataset.from_tensor_slices((feat_other_train, other_labels))

ds_scratch_test = tf.data.Dataset.from_tensor_slices((feat_scratch_test, scratch_labels_test))
ds_rest_test = tf.data.Dataset.from_tensor_slices((feat_rest_test, rest_labels_test))
ds_other_test = tf.data.Dataset.from_tensor_slices((feat_other_test, other_labels_test))

complete_training_set = ds_scratch_train.concatenate(ds_rest_train).concatenate(ds_other_train)
complete_test_set = ds_scratch_test.concatenate(ds_rest_test).concatenate(ds_other_test)

In [20]:
complete_training_set.take(1)

<_TakeDataset element_spec=(TensorSpec(shape=(80, 10), dtype=tf.float64, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>

In [32]:
batch_size = 64

norm_layer = Normalization()
norm_layer.adapt(complete_training_set.map(lambda x, _: x))

train_ds = complete_training_set.shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = complete_test_set.batch(batch_size).prefetch(tf.data.AUTOTUNE)

model = models.Sequential([
    layers.Input(shape=(80, 10)),       # timesteps=80, features=10
    norm_layer,
    layers.LSTM(64, return_sequences=True, dropout=0.3, recurrent_dropout=0.3),
    layers.LSTM(32, dropout=0.3, recurrent_dropout=0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(3, activation='softmax')  # 3 classes
])

model.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])
model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ normalization_4 (Normalization) │ (None, 80, 10)         │            21 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_11 (LSTM)                  │ (None, 80, 64)         │        19,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_12 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         1,056 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 32,792 (128.10 KB)

 Trainable params: 32,771 (128.01 KB)

 Non-trainable params: 21 (88.00 B)

In [33]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=tf.keras.callbacks.EarlyStopping(verbose=1, patience=3)
)

Epoch 1/30


c:\Users\Aryan M\AppData\Local\Programs\Python\Python39\lib\site-packages\keras\src\backend\tensorflow\nn.py:717: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


251/251 ━━━━━━━━━━━━━━━━━━━━ 21s 66ms/step - accuracy: 0.8118 - loss: 0.6663 - val_accuracy: 0.3331 - val_loss: 4.3702
Epoch 2/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - accuracy: 0.5225 - loss: 1.9664 - val_accuracy: 0.3331 - val_loss: 3.8311
Epoch 3/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - accuracy: 0.5351 - loss: 1.4881 - val_accuracy: 0.3333 - val_loss: 2.1362
Epoch 4/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - accuracy: 0.4960 - loss: 1.3282 - val_accuracy: 0.3331 - val_loss: 2.4682
Epoch 5/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - accuracy: 0.4879 - loss: 1.4843 - val_accuracy: 0.3331 - val_loss: 1.9223
Epoch 6/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 14s 55ms/step - accuracy: 0.3818 - loss: 1.2399 - val_accuracy: 0.3331 - val_loss: 1.1076
Epoch 7/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 14s 54ms/step - accuracy: 0.1601 - loss: 1.0681 - val_accuracy: 0.3331 - val_loss: 1.1080
Epoch 8/30
251/251 ━━━━━━━━━━━━━━━━━━━━ 14s 55ms/step - accuracy: 0.1489 - loss: 1.0450 - val_accurac